# Segment Effects

Tests the six segment hypotheses in the [analysis plan](../analysis_plan.md). The results feed the targeting rule for challenge questions 3 and 4, and H2 also answers questions 5 and 8.

Each hypothesis is tested with one regression that includes an interaction term. **The interaction coefficient is the difference in the email effect between the two segments.** Standard errors are robust (HC3). The Holm correction is applied after all six tests are run.

In [1]:
import pandas as pd
import statsmodels.formula.api as smf

df = pd.read_csv("../data/hillstrom.csv")
df["email"] = (df["segment"] != "No E-Mail").astype(int)

METRICS = ["spend", "conversion", "visit"]


def interaction_test(data, treatment, segment, labels):
    """Fit outcome ~ treatment * segment for each metric and report the effect in each segment and their difference.

    labels maps the segment values 0 and 1 to readable names.
    """
    rows = []
    for metric in METRICS:
        model = smf.ols(f"{metric} ~ {treatment} * {segment}", data).fit(cov_type="HC3")
        tests = {
            f"effect for {labels[0]}": f"{treatment} = 0",
            f"effect for {labels[1]}": f"{treatment} + {treatment}:{segment} = 0",
            "difference (interaction)": f"{treatment}:{segment} = 0",
        }
        for label, hypothesis in tests.items():
            test = model.t_test(hypothesis)
            low, high = test.conf_int(alpha=0.05)[0]
            rows.append({"metric": metric, "estimate": label, "value": float(test.effect[0]),
                         "low": low, "high": high, "p_value": float(test.pvalue)})
    return pd.DataFrame(rows).set_index(["metric", "estimate"])

## H1: Recency

**Hypothesis:** the email effect is larger for customers who last purchased 1 to 3 months ago than for those who last purchased 4 to 12 months ago. The Mens and Womens groups are pooled.

Model: `spend ~ email + recent + email:recent`, where `recent` = 1 for 1 to 3 months.

In [2]:
df["recent"] = (df["recency"] <= 3).astype(int)
H1_LABELS = {0: "4-12 months (recent = 0)", 1: "1-3 months (recent = 1)"}

counts = pd.crosstab(df["recent"].map(H1_LABELS), df["email"].map({0: "No email", 1: "Email"}), margins=True)
counts.index.name, counts.columns.name = "Last purchase", None
counts

,Email,No email,All
Last purchase,,,
1-3 months (recent = 1),14955,7438,22393
4-12 months (recent = 0),27739,13868,41607
All,42694,21306,64000


In [3]:
h1 = interaction_test(df, "email", "recent", H1_LABELS)
h1.round(4)

value     low    high  \
metric     estimate                                                      
spend      effect for 4-12 months (recent = 0)  0.5389  0.2923  0.7855   
           effect for 1-3 months (recent = 1)   0.7027  0.2696  1.1358   
           difference (interaction)             0.1638 -0.3346  0.6622   
conversion effect for 4-12 months (recent = 0)  0.0049  0.0034  0.0064   
           effect for 1-3 months (recent = 1)   0.0050  0.0022  0.0079   
           difference (interaction)             0.0001 -0.0031  0.0034   
visit      effect for 4-12 months (recent = 0)  0.0589  0.0525  0.0652   
           effect for 1-3 months (recent = 1)   0.0645  0.0543  0.0746   
           difference (interaction)             0.0056 -0.0063  0.0176   

                                                p_value  
metric     estimate                                      
spend      effect for 4-12 months (recent = 0)   0.0000  
           effect for 1-3 months (recent = 1)    0.0015  
           difference (interaction)              0.5194  
conversion effect for 4-12 months (recent = 0)   0.0000  
           effect for 1-3 months (recent = 1)    0.0006  
           difference (interaction)              0.9363  
visit      effect for 4-12 months (recent = 0)   0.0000  
           effect for 1-3 months (recent = 1)    0.0000  
           difference (interaction)              0.3578

**Reading the result:** the email raised spend for both groups: +$0.54 for customers who last purchased 4-12 months ago and +$0.70 for those who purchased 1-3 months ago. **The interaction (+$0.16, 95% CI -$0.33 to $0.66, p = 0.52) cannot show that the email worked better for recent customers,** because the two estimates overlap heavily and the interval includes zero. Conversion and visit point the same way. The Holm correction is applied after all six tests.